### Step 1 — Check storage for two training epochs

Check free storage before uploading two independently encoded MNIST training datasets. This step does not program the FPGA or change hardware state.

In [1]:
from pathlib import Path
import shutil

data_dir = Path("/home/xilinx/jupyter_notebooks/snn_sdsp")
assert data_dir.is_dir()

disk = shutil.disk_usage(data_dir)

print("Data directory:", data_dir)
print("Free storage (MiB):", round(disk.free / (1024 ** 2), 1))
print("Total storage (MiB):", round(disk.total / (1024 ** 2), 1))
print("Storage check completed")

Data directory: /home/xilinx/jupyter_notebooks/snn_sdsp
Free storage (MiB): 6802.6
Total storage (MiB): 14077.5
Storage check completed


### Step 2 — Verify both encoded training epochs

Check the uploaded ZIP archives, their manifests, and their SHA256 hashes before programming the FPGA. Each archive contains 60 ordered shards of 1,000 pre-encoded training images.

In [2]:
from pathlib import Path
import hashlib
import json
import zipfile

data_dir = Path("/home/xilinx/jupyter_notebooks/snn_sdsp")
epoch_paths = {
    1: data_dir / "mnist_train_epoch01.zip",
    2: data_dir / "mnist_train_epoch02.zip",
}
epoch_manifests = {}
epoch_hashes = {}

for epoch in (1, 2):
    path = epoch_paths[epoch]
    assert path.is_file(), f"Missing archive: {path}"

    digest = hashlib.sha256()
    with path.open("rb") as source:
        for chunk in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(chunk)
    epoch_hashes[epoch] = digest.hexdigest()

    with zipfile.ZipFile(path) as archive:
        manifest = json.loads(archive.read("manifest.json"))
        shards = manifest["shards"]
        expected_names = [
            f"shard_{number:03d}.npz" for number in range(60)
        ]

        assert manifest["format_version"] == 1
        assert manifest["split"] == "train"
        assert manifest["epoch"] == epoch
        assert manifest["image_count"] == 60000
        assert manifest["shard_size"] == 1000
        assert manifest["num_steps"] == 100
        assert abs(manifest["gain"] - 0.1) < 1e-9
        assert manifest["bitorder"] == "little"
        assert len(shards) == 60
        assert [item["name"] for item in shards] == expected_names
        assert all(item["count"] == 1000 for item in shards)
        assert all(
            item["first_position"] == number * 1000
            for number, item in enumerate(shards)
        )
        assert sum(manifest["class_counts"]) == 60000
        assert set(archive.namelist()) == set(
            expected_names + ["manifest.json"]
        )

    epoch_manifests[epoch] = manifest
    print(f"Epoch {epoch} file:", path)
    print("File size:", path.stat().st_size, "bytes")
    print("Shards:", len(shards))
    print("Images:", manifest["image_count"])
    print("Class counts:", manifest["class_counts"])
    print("Total input spikes:", manifest["total_input_spikes"])
    print("SHA256:", epoch_hashes[epoch])

assert epoch_manifests[1]["class_counts"] == epoch_manifests[2]["class_counts"]
assert epoch_manifests[1]["seed"] != epoch_manifests[2]["seed"]
assert epoch_hashes[1] != epoch_hashes[2]

print("Both training epoch archives passed metadata checks")

Epoch 1 file: /home/xilinx/jupyter_notebooks/snn_sdsp/mnist_train_epoch01.zip
File size: 80760349 bytes
Shards: 60
Images: 60000
Class counts: [5923, 6742, 5958, 6131, 5842, 5421, 5918, 6265, 5851, 5949]
Total input spikes: 61459290
SHA256: f50034e5ee87e61b804a956c61bc30fbad8427000a9dd275c2b3da641901a890
Epoch 2 file: /home/xilinx/jupyter_notebooks/snn_sdsp/mnist_train_epoch02.zip
File size: 80766508 bytes
Shards: 60
Images: 60000
Class counts: [5923, 6742, 5958, 6131, 5842, 5421, 5918, 6265, 5851, 5949]
Total input spikes: 61466611
SHA256: af822f326721df9d313cff7321a5170f45de96b01ce507f01949602909c39762
Both training epoch archives passed metadata checks


### Step 3 — Load fixed evaluation data and locate overlay files

Load the same pre-encoded MNIST test set used in the pilot. Confirm that the bitstream and hardware metadata files are present without programming the FPGA yet.

In [3]:
from pathlib import Path
import hashlib
import numpy as np

data_dir = Path("/home/xilinx/jupyter_notebooks/snn_sdsp")
fixed_eval_file = data_dir / "mnist_test1000_fixed_words.npz"
bit_file = data_dir / "snn_sdsp.bit"
hwh_file = data_dir / "snn_sdsp.hwh"

for path in (fixed_eval_file, bit_file, hwh_file):
    assert path.is_file(), f"Missing file: {path}"
    print(path.name, "size:", path.stat().st_size, "bytes")

with np.load(fixed_eval_file, allow_pickle=False) as data:
    fixed_eval_words = data["words"]
    fixed_eval_labels = data["labels"]
    fixed_eval_indices = data["indices"]
    fixed_eval_steps = int(data["num_steps"].item())
    fixed_eval_gain = float(data["gain"].item())
    fixed_eval_split = str(data["split"].item())
    fixed_eval_bitorder = str(data["bitorder"].item())

assert fixed_eval_words.shape == (1000, 100, 25)
assert fixed_eval_words.dtype == np.uint32
assert fixed_eval_labels.shape == (1000,)
assert len(np.unique(fixed_eval_indices)) == 1000
assert fixed_eval_steps == 100
assert np.isclose(fixed_eval_gain, 0.1)
assert fixed_eval_split == "test"
assert fixed_eval_bitorder == "little"

class_counts = np.bincount(
    fixed_eval_labels.astype(np.int32), minlength=10
)
assert np.array_equal(class_counts, np.full(10, 100))

fixed_eval_sha256 = hashlib.sha256(
    fixed_eval_file.read_bytes()
).hexdigest()

print("Evaluation words shape:", fixed_eval_words.shape)
print("Class counts:", class_counts.tolist())
print("Evaluation SHA256:", fixed_eval_sha256)
print("Fixed evaluation data and overlay files are ready")

mnist_test1000_fixed_words.npz size: 1372377 bytes
snn_sdsp.bit size: 4045700 bytes
snn_sdsp.hwh size: 271489 bytes
Evaluation words shape: (1000, 100, 25)
Class counts: [100, 100, 100, 100, 100, 100, 100, 100, 100, 100]
Evaluation SHA256: 2ade2e125d2d14257b72cbf7957de0d32c5e0879007bacce561242318f3d7e5e
Fixed evaluation data and overlay files are ready


### Step 4 — Program the FPGA for the full training run

Load the bitstream once to restore initial weights. Configure 100 time steps, calcium leak every 10 steps, disabled BIST, and teacher current 128 in Q9 format.

In [4]:
from pynq import Overlay, MMIO

if "full_overlay_programmed" in globals():
    raise RuntimeError(
        "The overlay was already programmed in this kernel. "
        "Do not rerun this cell during training."
    )

print("Programming FPGA...")
overlay = Overlay(str(bit_file), download=True)
snn = MMIO(0x43C00000, 0x10000)
dma = overlay.axi_dma_mm2s

ca_interval_steps = 10
bist_interval_steps = 0
teacher_current_q9 = 128

snn.write(0x10, ca_interval_steps)
snn.write(0x14, bist_interval_steps)
snn.write(0x18, teacher_current_q9)

control = snn.read(0x00)
status = snn.read(0x04)
n_cycles = snn.read(0x08)
ca_interval = snn.read(0x10)
bist_interval = snn.read(0x14)
teacher_current = snn.read(0x18)
dma_status = dma.mmio.read(0x04)

print("Overlay loaded:", overlay.is_loaded())
print(f"CONTROL: 0x{control:08X}")
print(f"SNN status: 0x{status:08X}")
print("N_CYCLES:", n_cycles)
print("CA interval:", ca_interval)
print("BIST interval:", bist_interval)
print("Teacher current Q9:", teacher_current)
print(f"DMA status: 0x{dma_status:08X}")
print("DMA running:", dma.sendchannel.running)

assert overlay.is_loaded()
assert control == 0
assert status & 0x01
assert n_cycles == 100
assert ca_interval == 10
assert bist_interval == 0
assert teacher_current == 128
assert dma.sendchannel.running
assert dma.recvchannel is None
assert (dma_status & 0x70) == 0

full_overlay_programmed = True
print("Full-training hardware configuration passed")

Programming FPGA...


Overlay loaded: True
CONTROL: 0x00000000
SNN status: 0x00000001
N_CYCLES: 100
CA interval: 10
BIST interval: 0
Teacher current Q9: 128
DMA status: 0x00000000
DMA running: True
Full-training hardware configuration passed


### Step 5 — Reuse the verified frame-transfer function

Load only the `run_frame` function definition from the saved pilot notebook. Run one inference frame with learning and teacher input disabled to verify the transfer path without changing weights.

In [5]:
import ast
import json
import time
from time import sleep, monotonic, perf_counter
from pynq import allocate

pilot_notebook = data_dir / "mnist_train1000.ipynb"
assert pilot_notebook.is_file(), f"Missing notebook: {pilot_notebook}"

with pilot_notebook.open("r", encoding="utf-8") as source:
    notebook_data = json.load(source)

definitions = []

for cell in notebook_data["cells"]:
    if cell.get("cell_type") != "code":
        continue

    source = cell.get("source", "")
    if isinstance(source, list):
        source = "".join(source)

    if "def run_frame" not in source:
        continue

    tree = ast.parse(source)
    definitions.extend(
        node
        for node in tree.body
        if isinstance(node, ast.FunctionDef)
        and node.name == "run_frame"
    )

assert len(definitions) == 1, (
    f"Expected one run_frame definition, found {len(definitions)}"
)

function_module = ast.Module(
    body=[definitions[0]],
    type_ignores=[],
)
ast.fix_missing_locations(function_module)
exec(
    compile(function_module, str(pilot_notebook), "exec"),
    globals(),
)

assert callable(run_frame)

probe = run_frame(
    fixed_eval_words[0],
    learn=False,
    teacher=False,
)

assert probe["accepted"] == 100
assert len(probe["spike_counts"]) == 10

print("Reused function from:", pilot_notebook)
print("Probe accepted steps:", probe["accepted"])
print("Probe output spike counts:", probe["spike_counts"])
print("Frame-transfer function is ready")

Reused function from: /home/xilinx/jupyter_notebooks/snn_sdsp/mnist_train1000.ipynb
Probe accepted steps: 100
Probe output spike counts: [6, 5, 6, 5, 5, 5, 6, 6, 6, 5]
Frame-transfer function is ready


### Step 6 — Measure the pre-training baseline

Evaluate all 1,000 fixed test images with learning and teacher input disabled. Keep the results in notebook memory so the exact same evaluation can be repeated after each training epoch.

In [6]:
import time
import numpy as np

assert full_overlay_programmed
assert snn.read(0x10) == 10
assert snn.read(0x14) == 0
assert snn.read(0x18) == 128
assert fixed_eval_words.shape == (1000, 100, 25)

def evaluate_fixed_set(name):
    counts = np.empty((1000, 10), dtype=np.int32)
    start_time = time.perf_counter()

    for index in range(1000):
        result = run_frame(
            fixed_eval_words[index],
            learn=False,
            teacher=False,
        )

        spike_counts = np.asarray(
            result["spike_counts"], dtype=np.int32
        )
        assert spike_counts.shape == (10,)
        assert result["accepted"] == 100

        counts[index] = spike_counts

        if (index + 1) % 100 == 0:
            print(
                f"{name}: evaluated {index + 1}/1000 images",
                flush=True,
            )

    silent = counts.sum(axis=1) == 0
    winners = counts.argmax(axis=1)
    correct = (~silent) & (winners == fixed_eval_labels)
    active_count = int((~silent).sum())

    summary = {
        "counts": counts,
        "silent": silent,
        "correct": correct,
        "strict_accuracy": float(correct.mean()),
        "argmax_accuracy": float(
            (winners == fixed_eval_labels).mean()
        ),
        "active_accuracy": (
            float(correct.sum() / active_count)
            if active_count else 0.0
        ),
        "total_spikes": int(counts.sum()),
        "elapsed_seconds": time.perf_counter() - start_time,
    }

    print(name, "strict accuracy:", round(summary["strict_accuracy"], 4))
    print(name, "argmax accuracy:", round(summary["argmax_accuracy"], 4))
    print(name, "active-image accuracy:", round(summary["active_accuracy"], 4))
    print(name, "silent images:", int(silent.sum()))
    print(name, "total output spikes:", summary["total_spikes"])
    print(name, "elapsed seconds:", round(summary["elapsed_seconds"], 2))
    return summary

baseline_eval = evaluate_fixed_set("Baseline")
print("Pre-training baseline completed")

Baseline: evaluated 100/1000 images
Baseline: evaluated 200/1000 images
Baseline: evaluated 300/1000 images
Baseline: evaluated 400/1000 images
Baseline: evaluated 500/1000 images
Baseline: evaluated 600/1000 images
Baseline: evaluated 700/1000 images
Baseline: evaluated 800/1000 images
Baseline: evaluated 900/1000 images
Baseline: evaluated 1000/1000 images
Baseline strict accuracy: 0.059
Baseline argmax accuracy: 0.063
Baseline active-image accuracy: 0.0761
Baseline silent images: 225
Baseline total output spikes: 26653
Baseline elapsed seconds: 14.79
Pre-training baseline completed


### Step 7 — Train the complete first epoch

Train all 60,000 MNIST training images from the first encoded epoch archive. Read and verify one shard at a time, preserve the stored shuffled order, and report progress every 1,000 images. Keep the FPGA programmed and do not reload the bitstream during training.

In [7]:
import hashlib
import io
import time
import zipfile
import numpy as np

assert baseline_eval["counts"].shape == (1000, 10)
assert snn.read(0x10) == 10
assert snn.read(0x14) == 0
assert snn.read(0x18) == 128

if "epoch_progress" in globals():
    raise RuntimeError(
        "Epoch training was already started. Do not rerun this cell."
    )

epoch_progress = {
    1: {
        "next_shard": 0,
        "completed": 0,
        "target_spikes": 0,
        "elapsed_seconds": 0.0,
        "failed": False,
    },
    2: {
        "next_shard": 0,
        "completed": 0,
        "target_spikes": 0,
        "elapsed_seconds": 0.0,
        "failed": False,
    },
}

def train_epoch_until(epoch, stop_shard):
    state = epoch_progress[epoch]
    manifest = epoch_manifests[epoch]

    if state["failed"]:
        raise RuntimeError(
            "A previous training call failed. Do not resume blindly."
        )
    if not (state["next_shard"] < stop_shard <= 60):
        raise ValueError(
            f"Invalid shard range: {state['next_shard']} to {stop_shard}"
        )
    if epoch == 2:
        assert epoch_progress[1]["next_shard"] == 60
        assert "epoch1_eval" in globals()

    start_time = time.perf_counter()

    try:
        with zipfile.ZipFile(epoch_paths[epoch]) as archive:
            for shard_number in range(state["next_shard"], stop_shard):
                entry = manifest["shards"][shard_number]
                payload = archive.read(entry["name"])

                assert hashlib.sha256(payload).hexdigest() == entry["sha256"]

                with np.load(io.BytesIO(payload), allow_pickle=False) as shard:
                    words = shard["words"]
                    labels = shard["labels"]
                    indices = shard["indices"]
                    stored_epoch = int(shard["epoch"].item())
                    stored_shard = int(shard["shard"].item())
                    stored_start = int(shard["start"].item())

                assert words.shape == (1000, 100, 25)
                assert words.dtype == np.uint32
                assert labels.shape == (1000,)
                assert indices.shape == (1000,)
                assert np.all(labels < 10)
                assert stored_epoch == epoch
                assert stored_shard == shard_number
                assert stored_start == shard_number * 1000

                for position in range(1000):
                    label = int(labels[position])
                    result = run_frame(
                        words[position],
                        label=label,
                        learn=True,
                        teacher=True,
                    )

                    assert result["accepted"] == 100
                    assert (result["control"] & 0x44) == 0x44

                    state["target_spikes"] += int(
                        result["spike_counts"][label]
                    )
                    state["completed"] += 1

                state["next_shard"] = shard_number + 1
                print(
                    f"Epoch {epoch}: trained "
                    f"{state['completed']}/60000 images",
                    flush=True,
                )

                del payload, words, labels, indices

    except BaseException:
        state["failed"] = True
        raise

    state["elapsed_seconds"] += time.perf_counter() - start_time

    print("Next shard:", state["next_shard"])
    print("Completed images:", state["completed"])
    print(
        "Mean target-neuron spikes so far:",
        round(state["target_spikes"] / state["completed"], 2),
    )
    print(
        "Training wall time so far (seconds):",
        round(state["elapsed_seconds"], 2),
    )

train_epoch_until(1, 60)
print("Epoch 1 training completed")

Epoch 1: trained 1000/60000 images
Epoch 1: trained 2000/60000 images
Epoch 1: trained 3000/60000 images
Epoch 1: trained 4000/60000 images
Epoch 1: trained 5000/60000 images
Epoch 1: trained 6000/60000 images
Epoch 1: trained 7000/60000 images
Epoch 1: trained 8000/60000 images
Epoch 1: trained 9000/60000 images
Epoch 1: trained 10000/60000 images
Epoch 1: trained 11000/60000 images
Epoch 1: trained 12000/60000 images
Epoch 1: trained 13000/60000 images
Epoch 1: trained 14000/60000 images
Epoch 1: trained 15000/60000 images
Epoch 1: trained 16000/60000 images
Epoch 1: trained 17000/60000 images
Epoch 1: trained 18000/60000 images
Epoch 1: trained 19000/60000 images
Epoch 1: trained 20000/60000 images
Epoch 1: trained 21000/60000 images
Epoch 1: trained 22000/60000 images
Epoch 1: trained 23000/60000 images
Epoch 1: trained 24000/60000 images
Epoch 1: trained 25000/60000 images
Epoch 1: trained 26000/60000 images
Epoch 1: trained 27000/60000 images
Epoch 1: trained 28000/60000 images
E

### Step 8 — Evaluate after the first epoch

Run inference on the same fixed 1,000-image test set with learning and teacher input disabled. Compare accuracy, silent images, and output spikes against the pre-training baseline before starting epoch 2.

In [8]:
assert epoch_progress[1]["next_shard"] == 60
assert epoch_progress[1]["completed"] == 60000
assert not epoch_progress[1]["failed"]
assert epoch_progress[2]["completed"] == 0

epoch1_eval = evaluate_fixed_set("After epoch 1")

print(
    "Strict accuracy before/after epoch 1:",
    round(baseline_eval["strict_accuracy"], 4),
    round(epoch1_eval["strict_accuracy"], 4),
)
print(
    "Accuracy change (percentage points):",
    round(
        100 * (
            epoch1_eval["strict_accuracy"]
            - baseline_eval["strict_accuracy"]
        ),
        2,
    ),
)
print(
    "Silent images before/after epoch 1:",
    int(baseline_eval["silent"].sum()),
    int(epoch1_eval["silent"].sum()),
)
print(
    "Total output spikes before/after epoch 1:",
    baseline_eval["total_spikes"],
    epoch1_eval["total_spikes"],
)

for digit in range(10):
    selected = fixed_eval_labels == digit
    print(
        f"Digit {digit}: "
        f"correct before={int(baseline_eval['correct'][selected].sum())}/100, "
        f"after={int(epoch1_eval['correct'][selected].sum())}/100; "
        f"silent before={int(baseline_eval['silent'][selected].sum())}/100, "
        f"after={int(epoch1_eval['silent'][selected].sum())}/100"
    )

print("First-epoch evaluation completed")

After epoch 1: evaluated 100/1000 images
After epoch 1: evaluated 200/1000 images
After epoch 1: evaluated 300/1000 images
After epoch 1: evaluated 400/1000 images
After epoch 1: evaluated 500/1000 images
After epoch 1: evaluated 600/1000 images
After epoch 1: evaluated 700/1000 images
After epoch 1: evaluated 800/1000 images
After epoch 1: evaluated 900/1000 images
After epoch 1: evaluated 1000/1000 images
After epoch 1 strict accuracy: 0.32
After epoch 1 argmax accuracy: 0.33
After epoch 1 active-image accuracy: 0.5161
After epoch 1 silent images: 380
After epoch 1 total output spikes: 4116
After epoch 1 elapsed seconds: 14.72
Strict accuracy before/after epoch 1: 0.059 0.32
Accuracy change (percentage points): 26.1
Silent images before/after epoch 1: 225 380
Total output spikes before/after epoch 1: 26653 4116
Digit 0: correct before=48/100, after=86/100; silent before=4/100, after=10/100
Digit 1: correct before=0/100, after=10/100; silent before=83/100, after=83/100
Digit 2: correc

### Step 9 — Train the complete second epoch

Continue training from the weights learned in epoch 1. Read all 60 shards from the independently shuffled and re-encoded second-epoch archive. Do not reprogram the FPGA between epochs.

In [9]:
assert epoch_progress[1]["next_shard"] == 60
assert epoch_progress[1]["completed"] == 60000
assert not epoch_progress[1]["failed"]
assert "epoch1_eval" in globals()

assert epoch_progress[2]["next_shard"] == 0
assert epoch_progress[2]["completed"] == 0

train_epoch_until(2, 60)

assert epoch_progress[2]["next_shard"] == 60
assert epoch_progress[2]["completed"] == 60000
assert not epoch_progress[2]["failed"]

print("Epoch 2 training completed")

Epoch 2: trained 1000/60000 images
Epoch 2: trained 2000/60000 images
Epoch 2: trained 3000/60000 images
Epoch 2: trained 4000/60000 images
Epoch 2: trained 5000/60000 images
Epoch 2: trained 6000/60000 images
Epoch 2: trained 7000/60000 images
Epoch 2: trained 8000/60000 images
Epoch 2: trained 9000/60000 images
Epoch 2: trained 10000/60000 images
Epoch 2: trained 11000/60000 images
Epoch 2: trained 12000/60000 images
Epoch 2: trained 13000/60000 images
Epoch 2: trained 14000/60000 images
Epoch 2: trained 15000/60000 images
Epoch 2: trained 16000/60000 images
Epoch 2: trained 17000/60000 images
Epoch 2: trained 18000/60000 images
Epoch 2: trained 19000/60000 images
Epoch 2: trained 20000/60000 images
Epoch 2: trained 21000/60000 images
Epoch 2: trained 22000/60000 images
Epoch 2: trained 23000/60000 images
Epoch 2: trained 24000/60000 images
Epoch 2: trained 25000/60000 images
Epoch 2: trained 26000/60000 images
Epoch 2: trained 27000/60000 images
Epoch 2: trained 28000/60000 images
E

### Step 10 — Evaluate after the second epoch

Evaluate the same fixed 1,000-image test set without learning or teacher input. Keep the baseline, epoch-1, and epoch-2 results separately for comparison.

In [10]:
assert epoch_progress[2]["next_shard"] == 60
assert epoch_progress[2]["completed"] == 60000
assert not epoch_progress[2]["failed"]

epoch2_eval = evaluate_fixed_set("After epoch 2")

for name, result in (
    ("Baseline", baseline_eval),
    ("After epoch 1", epoch1_eval),
    ("After epoch 2", epoch2_eval),
):
    print(
        f"{name}: strict accuracy={result['strict_accuracy']:.4f}, "
        f"silent images={int(result['silent'].sum())}, "
        f"total output spikes={result['total_spikes']}"
    )

print("Two-epoch evaluation completed")

After epoch 2: evaluated 100/1000 images
After epoch 2: evaluated 200/1000 images
After epoch 2: evaluated 300/1000 images
After epoch 2: evaluated 400/1000 images
After epoch 2: evaluated 500/1000 images
After epoch 2: evaluated 600/1000 images
After epoch 2: evaluated 700/1000 images
After epoch 2: evaluated 800/1000 images
After epoch 2: evaluated 900/1000 images
After epoch 2: evaluated 1000/1000 images
After epoch 2 strict accuracy: 0.363
After epoch 2 argmax accuracy: 0.383
After epoch 2 active-image accuracy: 0.5056
After epoch 2 silent images: 282
After epoch 2 total output spikes: 6486
After epoch 2 elapsed seconds: 14.73
Baseline: strict accuracy=0.0590, silent images=225, total output spikes=26653
After epoch 1: strict accuracy=0.3200, silent images=380, total output spikes=4116
After epoch 2: strict accuracy=0.3630, silent images=282, total output spikes=6486
Two-epoch evaluation completed


### Step 11 — Verify the third training epoch

Check the newly uploaded epoch-3 archive and register its ordered shards for training. This step does not program the FPGA or update weights.

In [11]:
import hashlib
import json
import zipfile

assert epoch_progress[2]["next_shard"] == 60
assert epoch_progress[2]["completed"] == 60000
assert not epoch_progress[2]["failed"]
assert "epoch2_eval" in globals()
assert overlay.is_loaded()

epoch3_path = data_dir / "mnist_train_epoch03.zip"
assert epoch3_path.is_file()

digest = hashlib.sha256()
with epoch3_path.open("rb") as source:
    for chunk in iter(lambda: source.read(1024 * 1024), b""):
        digest.update(chunk)
epoch3_hash = digest.hexdigest()

with zipfile.ZipFile(epoch3_path) as archive:
    epoch3_manifest = json.loads(archive.read("manifest.json"))
    shards = epoch3_manifest["shards"]
    expected_names = [
        f"shard_{number:03d}.npz" for number in range(60)
    ]

    assert epoch3_manifest["format_version"] == 1
    assert epoch3_manifest["split"] == "train"
    assert epoch3_manifest["epoch"] == 3
    assert epoch3_manifest["seed"] == 2030
    assert epoch3_manifest["image_count"] == 60000
    assert epoch3_manifest["shard_size"] == 1000
    assert epoch3_manifest["num_steps"] == 100
    assert abs(epoch3_manifest["gain"] - 0.1) < 1e-9
    assert epoch3_manifest["bitorder"] == "little"
    assert len(shards) == 60
    assert [item["name"] for item in shards] == expected_names
    assert all(item["count"] == 1000 for item in shards)
    assert all(
        item["first_position"] == number * 1000
        for number, item in enumerate(shards)
    )
    assert set(archive.namelist()) == set(
        expected_names + ["manifest.json"]
    )

assert epoch3_manifest["class_counts"] == epoch_manifests[1]["class_counts"]

epoch_paths[3] = epoch3_path
epoch_manifests[3] = epoch3_manifest
epoch_hashes[3] = epoch3_hash

print("Epoch 3 file:", epoch3_path)
print("File size:", epoch3_path.stat().st_size, "bytes")
print("Shards:", len(shards))
print("Images:", epoch3_manifest["image_count"])
print("Class counts:", epoch3_manifest["class_counts"])
print("Total input spikes:", epoch3_manifest["total_input_spikes"])
print("SHA256:", epoch3_hash)
print("Third training epoch passed metadata checks")

Epoch 3 file: /home/xilinx/jupyter_notebooks/snn_sdsp/mnist_train_epoch03.zip
File size: 80767671 bytes
Shards: 60
Images: 60000
Class counts: [5923, 6742, 5958, 6131, 5842, 5421, 5918, 6265, 5851, 5949]
Total input spikes: 61467643
SHA256: 06369517180ce20d85732e88f1ea9c7606538c56027bef165abe297af03b0e5a
Third training epoch passed metadata checks


### Step 12 — Train the complete third epoch

Continue from the weights learned in the first two epochs. Train all 60 ordered shards from the independently encoded third-epoch archive without reprogramming the FPGA.

In [12]:
assert epoch_progress[2]["next_shard"] == 60
assert epoch_progress[2]["completed"] == 60000
assert not epoch_progress[2]["failed"]
assert "epoch2_eval" in globals()
assert 3 in epoch_paths
assert 3 in epoch_manifests

if 3 in epoch_progress:
    raise RuntimeError(
        "Epoch 3 was already started. Do not run this cell twice."
    )

epoch_progress[3] = {
    "next_shard": 0,
    "completed": 0,
    "target_spikes": 0,
    "elapsed_seconds": 0.0,
    "failed": False,
}

train_epoch_until(3, 60)

assert epoch_progress[3]["next_shard"] == 60
assert epoch_progress[3]["completed"] == 60000
assert not epoch_progress[3]["failed"]

print("Epoch 3 training completed")

Epoch 3: trained 1000/60000 images
Epoch 3: trained 2000/60000 images
Epoch 3: trained 3000/60000 images
Epoch 3: trained 4000/60000 images
Epoch 3: trained 5000/60000 images
Epoch 3: trained 6000/60000 images
Epoch 3: trained 7000/60000 images
Epoch 3: trained 8000/60000 images
Epoch 3: trained 9000/60000 images
Epoch 3: trained 10000/60000 images
Epoch 3: trained 11000/60000 images
Epoch 3: trained 12000/60000 images
Epoch 3: trained 13000/60000 images
Epoch 3: trained 14000/60000 images
Epoch 3: trained 15000/60000 images
Epoch 3: trained 16000/60000 images
Epoch 3: trained 17000/60000 images
Epoch 3: trained 18000/60000 images
Epoch 3: trained 19000/60000 images
Epoch 3: trained 20000/60000 images
Epoch 3: trained 21000/60000 images
Epoch 3: trained 22000/60000 images
Epoch 3: trained 23000/60000 images
Epoch 3: trained 24000/60000 images
Epoch 3: trained 25000/60000 images
Epoch 3: trained 26000/60000 images
Epoch 3: trained 27000/60000 images
Epoch 3: trained 28000/60000 images
E

### Step 13 — Evaluate after the third epoch

Run inference on the same fixed 1,000-image test set with learning and teacher input disabled. Compare the baseline and all three epoch results without overwriting earlier measurements.

In [13]:
assert epoch_progress[3]["next_shard"] == 60
assert epoch_progress[3]["completed"] == 60000
assert not epoch_progress[3]["failed"]

epoch3_eval = evaluate_fixed_set("After epoch 3")

for name, result in (
    ("Baseline", baseline_eval),
    ("After epoch 1", epoch1_eval),
    ("After epoch 2", epoch2_eval),
    ("After epoch 3", epoch3_eval),
):
    print(
        f"{name}: "
        f"strict accuracy={result['strict_accuracy']:.4f}, "
        f"silent images={int(result['silent'].sum())}, "
        f"total output spikes={result['total_spikes']}"
    )

print(
    "Epoch 2 to epoch 3 accuracy change (percentage points):",
    round(
        100 * (
            epoch3_eval["strict_accuracy"]
            - epoch2_eval["strict_accuracy"]
        ),
        2,
    ),
)

for digit in range(10):
    selected = fixed_eval_labels == digit
    print(
        f"Digit {digit}: "
        f"correct epoch2={int(epoch2_eval['correct'][selected].sum())}/100, "
        f"epoch3={int(epoch3_eval['correct'][selected].sum())}/100; "
        f"silent epoch2={int(epoch2_eval['silent'][selected].sum())}/100, "
        f"epoch3={int(epoch3_eval['silent'][selected].sum())}/100"
    )

print("Three-epoch evaluation completed")

After epoch 3: evaluated 100/1000 images
After epoch 3: evaluated 200/1000 images
After epoch 3: evaluated 300/1000 images
After epoch 3: evaluated 400/1000 images
After epoch 3: evaluated 500/1000 images
After epoch 3: evaluated 600/1000 images
After epoch 3: evaluated 700/1000 images
After epoch 3: evaluated 800/1000 images
After epoch 3: evaluated 900/1000 images
After epoch 3: evaluated 1000/1000 images
After epoch 3 strict accuracy: 0.396
After epoch 3 argmax accuracy: 0.402
After epoch 3 active-image accuracy: 0.5045
After epoch 3 silent images: 215
After epoch 3 total output spikes: 8522
After epoch 3 elapsed seconds: 14.76
Baseline: strict accuracy=0.0590, silent images=225, total output spikes=26653
After epoch 1: strict accuracy=0.3200, silent images=380, total output spikes=4116
After epoch 2: strict accuracy=0.3630, silent images=282, total output spikes=6486
After epoch 3: strict accuracy=0.3960, silent images=215, total output spikes=8522
Epoch 2 to epoch 3 accuracy chang